<a href="https://colab.research.google.com/github/SristiSarkarMCKV/Yes-Bank-Data-science-project/blob/main/Yes_bank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')


# 1. Data Loading & Cleaning
def load_and_clean_data(filepath: str) -> pd.DataFrame:
  df = pd.read_csv(filepath)
  df['Date_dt'] = pd.to_datetime(df['Date'], format='%b-%y')
  df = df.sort_values('Date_dt').reset_index(drop=True)
  assert df.isnull().sum().sum() == 0, 'Missing values detected!'
  return df


# 2. Exploratory Data Analysis & Multicollinearity Check
def run_eda(df: pd.DataFrame):
  df['MA_6'] = df['Close'].rolling(window=6).mean()
  df['MA_12'] = df['Close'].rolling(window=12).mean()

  # VIF Calculation
  X_vif = df[['Open', 'High', 'Low']]
  vif_df = pd.DataFrame({
      'Feature': X_vif.columns,
      'VIF': [
          variance_inflation_factor(X_vif.values, i)
          for i in range(X_vif.shape[1])
      ],
  })
  print('Variance Inflation Factors:\n', vif_df, '\n')

  # Visualizations
  fig, axes = plt.subplots(2, 2, figsize=(16, 10))
  axes[0, 0].plot(df['Date_dt'], df['Close'], label='Close Price', linewidth=2)
  axes[0, 0].plot(
      df['Date_dt'], df['MA_6'], label='6-Month MA', linestyle='--'
  )
  axes[0, 0].plot(
      df['Date_dt'], df['MA_12'], label='12-Month MA', linestyle='-.'
  )
  axes[0, 0].axvline(
      pd.to_datetime('2018-09-01'),
      color='red',
      linestyle=':',
      label='Sep 2018 Crisis',
  )
  axes[0, 0].set_title('Yes Bank Closing Price & Moving Averages')
  axes[0, 0].legend()

  sns.heatmap(
      df[['Open', 'High', 'Low', 'Close']].corr(),
      annot=True,
      cmap='Blues',
      fmt='.4f',
      ax=axes[0, 1],
  )
  axes[0, 1].set_title('Correlation Heatmap')

  sns.boxplot(data=df[['Open', 'High', 'Low', 'Close']], ax=axes[1, 0])
  axes[1, 0].set_title('Price Distributions & Outliers')

  sns.regplot(
      x='Open', y='Close', data=df, ax=axes[1, 1], line_kws={'color': 'red'}
  )
  axes[1, 1].set_title('Open vs Close Linear Fit')
  plt.tight_layout()
  plt.show()


# 3. Preprocessing & Chronological Train-Test Split
def prepare_data(df: pd.DataFrame, split_ratio: float = 0.80):
  features = ['Open', 'High', 'Low']
  X, y = df[features], df['Close']
  split_idx = int(len(df) * split_ratio)

  X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
  y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
  dates_test = df['Date_dt'].iloc[split_idx:]

  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)
  return X_train_scaled, X_test_scaled, y_train, y_test, dates_test, features


# 4. Model Training, Hyperparameter Tuning & Evaluation
def train_and_evaluate(X_train, X_test, y_train, y_test):
  tscv = TimeSeriesSplit(n_splits=5)
  model_configs = {
      'Linear Regression': (LinearRegression(), {}),
      'Ridge': (Ridge(), {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}),
      'Lasso': (
          Lasso(max_iter=10000),
          {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]},
      ),
      'ElasticNet': (
          ElasticNet(max_iter=10000),
          {'alpha': [0.001, 0.01, 0.1, 1.0], 'l1_ratio': [0.1, 0.5, 0.9]},
      ),
      'Random Forest': (
          RandomForestRegressor(random_state=42),
          {'n_estimators': [100, 200], 'max_depth': [None, 5, 10]},
      ),
      'Gradient Boosting': (
          GradientBoostingRegressor(random_state=42),
          {
              'n_estimators': [100, 200],
              'learning_rate': [0.05, 0.1],
              'max_depth': [3, 5],
          },
      ),
  }

  results, trained_models = [], {}
  for name, (model, params) in model_configs.items():
    if params:
      grid = GridSearchCV(
          model, params, cv=tscv, scoring='neg_mean_squared_error'
      )
      grid.fit(X_train, y_train)
      best_model = grid.best_estimator_
      best_params = grid.best_params_
    else:
      best_model = model.fit(X_train, y_train)
      best_params = 'Default'

    trained_models[name] = best_model
    preds = best_model.predict(X_test)
    results.append({
        'Model': name,
        'Best Params': str(best_params),
        'MSE': round(mean_squared_error(y_test, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 4),
        'MAE': round(mean_absolute_error(y_test, preds), 4),
        'MAPE (%)': round(mean_absolute_percentage_error(y_test, preds) * 100, 2),
        'R2': round(r2_score(y_test, preds), 4),
    })

  results_df = pd.DataFrame(results).sort_values(by='RMSE')
  print(results_df.to_string(index=False))
  return trained_models, results_df


# Execute Pipeline
df = load_and_clean_data('data_YesBank_StockPrices.csv')
run_eda(df)
X_train, X_test, y_train, y_test, dates_test, feat_names = prepare_data(df)
models, summary_table = train_and_evaluate(X_train, X_test, y_train, y_test)